# Video Editing with Layout-Guided Attention (LTX)

SDEdit + layout-guided optimization using the LTX multi-stage pipeline.
The model is loaded once; re-run config and run cells to iterate.

In [1]:
import logging
import os
import sys
from pathlib import Path

import torch
from loguru import logger
from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
from ltx_pipelines.utils.constants import DEFAULT_NEGATIVE_PROMPT
from ltx_pipelines.utils.media_io import encode_video
from transformers import AutoTokenizer

PROJECT_ROOT = Path(os.getcwd()).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ltx.pipeline import (
    DiffusersCallbackAdapter,
    load_multi_stage_pipeline,
)
from src.optimization import (
    AttentionStore,
    GuidingPhrase,
    LatentRefiner,
    LayoutKLLoss,
    OptimizerType,
    get_token_indices_for_phrase_indices,
)
from src.video_utils import (
    SaveAttnMapsCallbackLTX,
    downsample_mask_to_latent_resolution,
    load_video_as_tensor,
)

logging.getLogger("imageio_ffmpeg").setLevel(logging.ERROR)
logging.getLogger("imageio").setLevel(logging.ERROR)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATA_PATH = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_PATH = os.path.join(PROJECT_ROOT, "outputs")
MODELS_PATH = os.path.join(PROJECT_ROOT, "models")

pipe = load_multi_stage_pipeline(
    checkpoint_path=os.path.join(MODELS_PATH, "ltx-2.3-22b-dev.safetensors"),
    spatial_upsampler_path=os.path.join(MODELS_PATH, "ltx-2.3-spatial-upscaler-x2-1.0.safetensors"),
    distilled_lora_path=os.path.join(MODELS_PATH, "ltx-2.3-22b-distilled-lora-384.safetensors"),
)

/home/user/LSJ/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Editing Functions

In [2]:
def setup_layout_guidance(
    bbox_paths,
    phrase_indices,
    prompt,
    tokenizer_path,
    height,
    width,
    num_frames,
    num_refinements,
    lr_start,
    lr_end,
    num_stages=2,
):
    """Build layout-guidance components: guiding phrases, bounding boxes, loss, and refiner."""
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    indices = get_token_indices_for_phrase_indices(phrase_indices, prompt, tokenizer)
    del tokenizer
    torch.cuda.empty_cache()

    words = prompt.split()
    guiding_phrases = [GuidingPhrase(" ".join(words[i] for i in group), idx) for group, idx in zip(phrase_indices, indices)]
    logger.info(f"Guiding phrases: {guiding_phrases}")

    bboxes = torch.cat([load_video_as_tensor(p, height=height, width=width, mode="L")[:, :, :num_frames] for p in bbox_paths])
    vae_spatial_scale_factor = 2 ** (num_stages - 1) * 32
    bboxes = downsample_mask_to_latent_resolution(bboxes, 8, vae_spatial_scale_factor)
    bboxes = (bboxes > 0.5).float()
    logger.info(f"Bboxes shape: {bboxes.shape}")

    refiner = LatentRefiner(
        loss_fn=LayoutKLLoss(bboxes=bboxes, indices=indices, eps=1e-8),
        num_refinements=num_refinements,
        lr_start=lr_start,
        lr_end=lr_end,
        betas=(0.9, 0.999),
        optimizer_type=OptimizerType.ADAMW8BIT,
    )
    return guiding_phrases, refiner


def edit_video(
    pipe,
    prompt,
    negative_prompt,
    source_video_path,
    height,
    width,
    num_frames,
    frame_rate,
    num_inference_steps,
    cfg_guidance_scale,
    seed,
    denoise_from,
    guiding_phrases,
    refiner,
    output_path,
    num_stages=2,
    layers_to_hook=None,
    refine_steps=None,
    phrase_indices=None,
    tiling_config=None,
):
    """Edit a source video using SDEdit + layout guidance via the multi-stage pipeline."""
    layers_to_hook = layers_to_hook or [8, 9, 10]
    refine_steps = refine_steps or []

    store = AttentionStore()

    downscale = 2 ** (num_stages - 1)
    ltx_callback = SaveAttnMapsCallbackLTX(
        store=store,
        guiding_phrases=guiding_phrases,
        video_shape=(num_frames, height // downscale, width // downscale),
        output_path=str(Path(output_path) / "attention_maps"),
        fps=3.0,
    )
    callback = DiffusersCallbackAdapter(ltx_callback)

    logger.info(f"SDEdit from step {denoise_from}/{num_inference_steps}, {num_inference_steps - denoise_from} denoise steps")

    stage_results = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_frames=num_frames,
        frame_rate=frame_rate,
        num_inference_steps=num_inference_steps,
        cfg_guidance_scale=cfg_guidance_scale,
        seed=seed,
        num_stages=num_stages,
        tiling_config=tiling_config,
        attention_store=store,
        layers_to_hook=layers_to_hook,
        refiner=refiner,
        refine_steps=refine_steps,
        callback=callback,
        phrase_indices=phrase_indices,
        source_video_path=source_video_path,
        denoise_from=denoise_from,
    )

    return stage_results

## Configuration

In [ ]:
# Provide your own source video and layout mask
source_video_path = "/home/user/LSJ/inputs/editing/dog-running/dog-running.mp4"
mask_path = "/home/user/LSJ/inputs/editing/dog-running/masks.mp4"
output_path = "/home/user/LSJ/inputs/editing/dog-running/"

gemma_root_path = os.path.join(MODELS_PATH, "gemma-3-12b-it-qat-q4_0-unquantized")

prompt = "A cheetah is running across the grass with a stick in its mouth."
negative_prompt = DEFAULT_NEGATIVE_PROMPT
phrase_indices = [[1]]

num_frames = 81
frame_rate = 24.0

height = 448
width = 704
num_stages = 2
num_inference_steps = 40
cfg_guidance_scale = 7.0
denoise_from = 10
layers_to_hook = [8, 9, 10]
num_refinements = 1
lr_start = 0.01
lr_end = 0.01
refine_steps = list(range(num_inference_steps - denoise_from))

tiling_config = TilingConfig.default()

Path(output_path).mkdir(parents=True, exist_ok=True)

guiding_phrases, refiner = setup_layout_guidance(
    bbox_paths=[mask_path],
    phrase_indices=phrase_indices,
    prompt=prompt,
    tokenizer_path=gemma_root_path,
    height=height,
    width=width,
    num_frames=num_frames,
    num_refinements=num_refinements,
    lr_start=lr_start,
    lr_end=lr_end,
    num_stages=num_stages,
)

2026-03-08 15:39:17.252 | INFO     | __main__:setup_layout_guidance:25 - Guiding phrases: [GuidingPhrase(phrase='cheetah', indices=[2])]
2026-03-08 15:39:22.461 | INFO     | __main__:setup_layout_guidance:33 - Bboxes shape: torch.Size([1, 1, 11, 7, 11])


## Edit Source Video

In [6]:
seed = 42

stage_results = edit_video(
    pipe=pipe,
    prompt=prompt,
    negative_prompt=negative_prompt,
    source_video_path=source_video_path,
    height=height,
    width=width,
    num_frames=num_frames,
    frame_rate=frame_rate,
    num_inference_steps=num_inference_steps,
    cfg_guidance_scale=cfg_guidance_scale,
    seed=seed,
    denoise_from=denoise_from,
    guiding_phrases=guiding_phrases,
    refiner=refiner,
    output_path=output_path,
    num_stages=num_stages,
    layers_to_hook=layers_to_hook,
    refine_steps=refine_steps,
    phrase_indices=phrase_indices,
    tiling_config=tiling_config,
)

video_chunks_number = get_video_chunks_number(num_frames, tiling_config)
for i, (video, audio) in enumerate(stage_results, 1):
    stage_path = Path(output_path) / f"stage_{i}.mp4"
    encode_video(
        video=video,
        fps=frame_rate,
        audio=audio,
        output_path=stage_path,
        video_chunks_number=video_chunks_number,
    )
    logger.success(f"Stage {i} saved: {stage_path}")

2026-03-08 15:39:23.729 | INFO     | src.video_utils:__init__:85 - Saving attention maps to /home/user/LSJ/inputs/editing/dog-running/attention_maps
2026-03-08 15:39:23.730 | INFO     | __main__:edit_video:84 - SDEdit from step 6/40, 34 denoise steps
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[Stage 1/2] Generating at 352x224


2026-03-08 15:39:39.225 | INFO     | src.ltx.attn_processor:register_attention_hook:56 - Registered 3/48 transformer blocks
Denoising: 100%|██████████| 34/34 [00:26<00:00,  1.28it/s, grad=2.69e-02, loss=3.07→3.07, lr=1.00e-02, refine=1/1]
2026-03-08 15:40:06.708 | INFO     | src.ltx.attn_processor:unregister_attention_hook:69 - Unregistered 3 attention hooks


[Stage 2/2] Upscaling to 704x448


100%|██████████| 2/2 [00:00<00:00,  8.39it/s]
2026-03-08 15:40:21.703 | SUCCESS  | __main__:<module>:36 - Stage 1 saved: /home/user/LSJ/inputs/editing/dog-running/stage_1.mp4
100%|██████████| 2/2 [00:00<00:00,  2.71it/s]
2026-03-08 15:40:24.436 | SUCCESS  | __main__:<module>:36 - Stage 2 saved: /home/user/LSJ/inputs/editing/dog-running/stage_2.mp4
